# SAM3만 활용해서 이상 탐지

**목표:** 별도 YOLO 학습 없이 SAM3 텍스트 프롬프트만으로 엘리베이터 계단 노란색 세이프가드의 이상(파손·누락)을 탐지한다.

**핵심 아이디어:**
1. 프레임 하단에 가상의 가로선을 긋는다.
2. SAM3가 `"yellow bar"` 텍스트로 전체 이미지에서 노란 막대를 세그먼트한다.
3. 반환된 마스크 중 기준선 y에 픽셀이 존재하는 것만 남긴다 (선에 닿은 객체만 카운트).
4. 정상 영상 기준 평균/표준편차로 임계값을 설정 → 감지 수 미달이면 이상 판정.

> `SAM3SemanticPredictor`는 텍스트 전용 프리딕터이므로 point prompt는 지원하지 않는다.
> 대신 마스크 후처리 단계에서 기준선 교차 여부로 필터링한다.

In [ ]:
import cv2
import numpy as np
from pathlib import Path
from ultralytics.models.sam.predict import SAM3SemanticPredictor

In [ ]:
VIDEO_PATH   = "videos/test.mp4"
OUTPUT_VIDEO = "sam3_anomaly_output.mp4"

# SAM3SemanticPredictor 는 텍스트 전용 — 짧고 단순한 영어 단어가 잘 매칭됨
TEXT_PROMPT  = "yellow bar"

# 기준선: 이 y 위치에 마스크 픽셀이 있으면 '선에 닿은 객체'로 카운트
LINE_Y_RATIO = 0.85   # 프레임 높이의 85% 지점

SIGMA_MULT   = 2.0    # 임계값 = mean - SIGMA_MULT * std
CONF         = 0.25

## Step 1 — 프레임 로드 & 기준선 미리보기

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
frames = []
while True:
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(frame)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
cap.release()

fh, fw = frames[0].shape[:2]
line_y = int(fh * LINE_Y_RATIO)

print(f"총 {len(frames)} 프레임  ({fw}×{fh})  FPS={fps:.1f}")
print(f"기준선 y={line_y}px  (전체 높이의 {LINE_Y_RATIO*100:.0f}%)")

preview = frames[0].copy()
cv2.line(preview, (0, line_y), (fw - 1, line_y), (0, 255, 255), 3)
cv2.putText(preview, f"filter line  y={line_y}", (20, line_y - 15),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, (0, 255, 255), 2)
cv2.imwrite("sam3_preview_line.jpg", preview)
print("기준선 미리보기 저장: sam3_preview_line.jpg")

## Step 2 — SAM3 세그먼트 + 기준선 교차 필터링

각 프레임에서 SAM3가 찾은 마스크 중 `line_y` 행에 픽셀이 1개 이상 있는 것만 카운트한다.

In [ ]:
overrides = dict(
    conf=CONF,
    task="segment",
    mode="predict",
    imgsz=640,
    model="sam3.pt",
    half=True,
    save=False,
)
predictor = SAM3SemanticPredictor(overrides=overrides)

seg_counts  = []   # 기준선에 닿은 마스크 수 (프레임별)
all_results = []   # Step 4에서 재사용하기 위해 결과 캐싱

for frame_idx, frame in enumerate(frames):
    predictor.set_image(frame)
    r = predictor(text=[TEXT_PROMPT])[0]

    hit_masks = []   # 기준선에 닿은 마스크만

    if r.masks is not None:
        for mask in r.masks.data.cpu().numpy():
            # 마스크를 원본 해상도로 리사이즈
            mask_r = cv2.resize(mask, (fw, fh), interpolation=cv2.INTER_NEAREST)
            # 기준선 행에 양수 픽셀이 있으면 '선에 닿음'
            if mask_r[line_y, :].max() > 0:
                hit_masks.append(mask_r)

    seg_counts.append(len(hit_masks))
    all_results.append((r.orig_img, hit_masks))

    if frame_idx % 30 == 0:
        total = len(r.masks.data) if r.masks is not None else 0
        print(f"  frame {frame_idx:4d}  전체={total}  기준선 교차={len(hit_masks)}")

seg_counts = np.array(seg_counts)
print(f"\n완료  min={seg_counts.min()}  max={seg_counts.max()}  "
      f"mean={seg_counts.mean():.1f}  std={seg_counts.std():.1f}")

## Step 3 — 임계값 계산

In [ ]:
threshold = max(0.0, seg_counts.mean() - SIGMA_MULT * seg_counts.std())
print(f"감지 분포: min={seg_counts.min()}  max={seg_counts.max()}  "
      f"mean={seg_counts.mean():.1f}  std={seg_counts.std():.1f}")
print(f"임계값: {threshold:.2f}  (mean - {SIGMA_MULT}σ)")
print(f"  → 기준선 교차 마스크 수 < {threshold:.2f} 이면 이상 판정")

## Step 4 — 이상 탐지 결과 영상 출력

Step 2에서 캐싱한 결과를 재사용하므로 SAM3를 다시 돌리지 않는다.

In [ ]:
writer = cv2.VideoWriter(
    OUTPUT_VIDEO,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (fw, fh),
)

STATUS_COLOR = {"OK": (0, 200, 0), "BROKEN": (0, 0, 255), "MISSING": (0, 0, 255)}
anomaly_frames = []

for frame_idx, (orig_img, hit_masks) in enumerate(all_results):
    count = len(hit_masks)

    if count == 0:
        status = "MISSING"
    elif count < threshold:
        status = "BROKEN"
    else:
        status = "OK"

    out_frame = orig_img.copy()
    color = STATUS_COLOR[status]

    # 기준선에 닿은 마스크만 오버레이
    for mask_r in hit_masks:
        overlay = out_frame.copy()
        overlay[mask_r > 0] = (0, 215, 255)
        cv2.addWeighted(overlay, 0.35, out_frame, 0.65, 0, out_frame)

    # 기준선
    cv2.line(out_frame, (0, line_y), (fw - 1, line_y), (0, 255, 255), 2)

    # 상단 HUD
    hud = out_frame.copy()
    cv2.rectangle(hud, (0, 0), (fw, 65), (0, 0, 0), -1)
    cv2.addWeighted(hud, 0.45, out_frame, 0.55, 0, out_frame)
    label = f"[{status}]  guards={count}  thr={threshold:.1f}  frame={frame_idx}"
    cv2.putText(out_frame, label, (12, 48),
                cv2.FONT_HERSHEY_SIMPLEX, 1.1, color, 2, cv2.LINE_AA)

    if status != "OK":
        cv2.rectangle(out_frame, (0, 0), (fw - 1, fh - 1), color, 6)
        anomaly_frames.append((frame_idx, status, count))

    writer.write(out_frame)

    if frame_idx % 50 == 0:
        print(f"  frame {frame_idx}  {status}  guards={count}")

writer.release()
print(f"\n검사 완료: 전체 {len(all_results)}프레임  이상 {len(anomaly_frames)}개")
print(f"출력 영상: {OUTPUT_VIDEO}")

## Step 5 — 이상 구간 요약

In [ ]:
if not anomaly_frames:
    print("이상 없음 — 모든 프레임 정상")
else:
    print(f"{'프레임':>8}  {'상태':>8}  {'감지수':>6}")
    print("-" * 30)
    for fidx, st, cnt in anomaly_frames:
        print(f"{fidx:>8}  {st:>8}  {cnt:>6}")
    print(f"\n이상률: {len(anomaly_frames)/len(all_results)*100:.1f}%")